# 🧪 Monte Carlo Simulation of Noise-Resilient Two-Qubit Variational Quantum Classifiers with Zero-Noise Extrapolation for Intent Classification

### CSC 133 — Modelling and Simulation

**Author:** John-Ronan S. Beira

---

**Keywords**

| Area | Keywords |
|---|---|
| **Model** | `Variational Quantum Classifier (VQC)` · `Quantum Machine Learning (QML)` |
| **Error Mitigation** | `Zero-Noise Extrapolation (ZNE)` |
| **Simulation** | `Monte Carlo Simulation` · `Noisy Intermediate-Scale Quantum (NISQ)` |
| **Noise and Sampling** | `Depolarizing Noise` · `Measurement Shots` |
| **Task** | `Intent Classification` · `Tanaos Dataset` |
| **Feature Processing** | `TF-IDF` · `Principal Component Analysis (PCA)` |

---

## 📌 Notebook Purpose

This notebook supports the research paper by implementing a reproducible simulation workflow for a **two-qubit Variational Quantum Classifier (VQC)**.

The experiment focuses on binary intent classification using the **Greeting** and **Farewell** classes from the **Tanaos synthetic intent classifier dataset**.

The notebook has two main parts:

| Part | Purpose |
|---|---|
| **Part 0 — Onboarding and Reproducibility Check** | Checks whether the local Jupyter environment, source-code imports, and project structure are working correctly. |
| **Part 1 — Research Experiment** | Runs the actual simulation workflow: dataset loading, TF-IDF, PCA, VQC construction, baseline evaluation, noisy simulation, ZNE, Monte Carlo trials, and result summaries. |

This structure lets the notebook work both as a tester onboarding file and as the main execution notebook for the study.

## 🗺️ Notebook Outline

### Part 0 — Onboarding and Reproducibility Check

| Step | Section | Purpose |
|---:|---|---|
| 0.1 | **Project Structure Check** | Confirms that the notebook can locate the repository source directory. |
| 0.2 | **Autoreload Setup** | Reloads local Python files automatically during development. |
| 0.3 | **Smoke Test** | Runs small test functions to confirm that imports from `src/` are working. |
| 0.4 | **Dependency Notes** | Summarizes the main tools used by the project environment. |

---

### Part 1 — Research Experiment

| Step | Section | Purpose |
|---:|---|---|
| 1.1 | **Research Configuration** | Defines labels, seeds, shot counts, noise levels, Monte Carlo trials, and experiment settings. |
| 1.2 | **Dataset Loading** | Loads the Tanaos dataset and filters it to the Greeting and Farewell classes. |
| 1.3 | **Text Preprocessing** | Prepares the utterances before feature extraction. |
| 1.4 | **Feature Extraction** | Converts text into TF-IDF vectors and reduces them to two PCA components. |
| 1.5 | **VQC Architecture** | Builds the two-qubit VQC using `ZZFeatureMap` and `RealAmplitudes`. |
| 1.6 | **Baseline Statevector Evaluation** | Evaluates the model without noise to establish a clean baseline. |
| 1.7 | **Noisy Simulation** | Evaluates the model under depolarizing noise and finite measurement shots. |
| 1.8 | **Zero-Noise Extrapolation** | Estimates zero-noise behavior from scaled noisy evaluations. |
| 1.9 | **Monte Carlo Trials** | Repeats the noisy evaluation across independent trials to measure accuracy stability. |
| 1.10 | **Results Summary** | Reports mean accuracy, variance, and comparisons across noise and shot settings. |

# 🧭 Part 0 — Notebook Onboarding and Reproducibility Check

Before running the research experiment, this section verifies that the local notebook environment is ready.

| Check | What it confirms |
|---|---|
| `src/` path access | The notebook can locate local project files. |
| Local imports | Python modules under `src/` can be imported. |
| Autoreload | Changes to `.py` files are picked up during development. |
| Smoke tests | Basic helper functions run without import errors. |

> 🔁 `autoreload` is enabled below so changes made to `.py` files can be used without restarting the kernel.

---

## 0.1 Project Structure and Import Check

This step prepares the notebook to work with local Python files.

| Item | Purpose |
|---|---|
| `notebooks/` | Contains the notebook used to run and inspect the experiment. |
| `src/` | Stores reusable Python modules for data loading, feature processing, and model logic. |
| Python import path | Allows the notebook to import project modules from `src/`. |

The next cell creates the local `src/` directory if needed and adds the notebook directory to Python’s import path.

In [ ]:
# Cell 1: Universal Path Setup
import os
import sys

# Get the directory where THIS notebook is located
nb_dir = os.getcwd()

print(f"Notebook is located at: {nb_dir}")

# Define the src path relative to this notebook
# If submitting a single file, we keep src in the same folder
src_path = os.path.join(nb_dir, "src")
os.makedirs(src_path, exist_ok=True)

# Add it to sys.path so 'import src' always works
if src_path not in sys.path:
    sys.path.append(nb_dir) 

print(f"✅ Source directory set to: {src_path}")
print(f"✅ System path updated. Imports will look in: {nb_dir}")

---

## 0.2 Autoreload Setup

This step enables automatic reloading of local Python modules.

When the code inside `src/` changes, Jupyter may continue using the old imported version. Autoreload reduces that friction during development.

| Tool | Role |
|---|---|
| `%load_ext autoreload` | Loads the autoreload extension. |
| `%autoreload 2` | Reloads imported modules before running code cells. |

> This is useful while editing `.py` files and testing them from the notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Use SVG output for high-quality inline Matplotlib figures in Jupyter.
from matplotlib_inline.backend_inline import set_matplotlib_formats
import matplotlib.pyplot as plt

set_matplotlib_formats("svg")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.format"] = "svg"
plt.rcParams["savefig.bbox"] = "tight"

---

## 0.3 Smoke Test

This step checks whether local module creation, import, and execution are working.

| Test | Expected result |
|---|---|
| Create `src/hello.py` | A small test module is written under `src/`. |
| Import module | `src.hello` imports without errors. |
| Call function | A simple function runs from the imported module. |
| Input test | The notebook accepts and echoes basic user input. |

If these tests pass, the notebook environment is ready for the research modules.

In [ ]:
%%writefile src/hello.py
def hello_world():
    print("Hello, World!")


def echo(value):
    return value

### 0.3.1 Import Test

The next cell imports the test module from `src/`.

**Expected result:** the import runs without an error.

In [ ]:
import src.hello

### 0.3.2 Function Test

The next cell calls a simple function from the imported module.

**Expected result:** the notebook prints a short confirmation message.

In [ ]:
src.hello.hello_world()

### 0.3.3 Basic Input Test

The next cell checks whether the notebook can accept simple user input.

**Expected result:** the notebook echoes the value entered by the user.

> This test is only for notebook interaction. It is not part of the research experiment.

In [ ]:
user_input = input("Enter something: ")
result = src.hello.echo(user_input)
print(f"You entered: {result}")
# If you're able to see your input printed then you're good to go!

# Dependencies

This document lists the project dependencies defined in the actual `Pipfile` for transparency and reproducibility.

---

## Package Descriptions

| Package | Purpose |
|---|---|
| `jupyter` | Interactive notebook environment used to run and document the experiment |
| `ipykernel` | Python kernel support for running the notebook |
| `nbstripout` | Removes notebook outputs before commits to keep version control clean |
| `pre-commit` | Git hook manager for automated code and notebook checks |
| `nbdime` | Diff and merge tools for Jupyter notebooks |
| `nbconvert[webpdf]` | Converts notebooks into export formats, including web-based PDF export |
| `playwright` | Browser automation backend required for web PDF export through `nbconvert` |
| `pandas` | Data manipulation, tabular summaries, and CSV output handling |
| `numpy` | Numerical computing, array operations, and reproducible random seeding |
| `scikit-learn` | TF-IDF vectorization, PCA dimensionality reduction, train-test splitting, and evaluation metrics |
| `matplotlib` | Plotting and figure generation for experimental results |
| `seaborn` | Statistical visualization support for plots and heatmaps |
| `datasets` | HuggingFace dataset loading for the Tanaos synthetic intent classifier dataset |
| `nltk` | Natural language processing utilities |
| `qiskit` | Core quantum computing SDK used for circuits and quantum simulation workflow |
| `qiskit-machine-learning` | Quantum machine learning components, including VQC-related tools |
| `qiskit-algorithms` | Quantum algorithm primitives and optimizer-related utilities |
| `pylatexenc` | LaTeX rendering support for Qiskit circuit drawings |
| `qiskit-aer` | Local quantum simulation backend, including noisy simulator support |

---

## Pipenv (`Pipfile`)

```toml
[[source]]
url = "https://pypi.org/simple"
verify_ssl = true
name = "pypi"

[packages]
jupyter = "*"
ipykernel = "*"
nbstripout = "*"
pre-commit = "*"
nbdime = "*"
nbconvert = {extras = ["webpdf"], version = "*"}
playwright = "==1.56.0"
pandas = "*"
numpy = "*"
scikit-learn = "*"
matplotlib = "*"
seaborn = "*"
datasets = "*"
nltk = "*"
qiskit = "*"
qiskit-machine-learning = "*"
qiskit-algorithms = "*"
pylatexenc = "*"
qiskit-aer = "*"

[dev-packages]

[requires]
python_version = "3.14"
```

> Install with: `pipenv install`

---

> **Python Version:** 3.14  
> **Package Index:** [https://pypi.org/simple](https://pypi.org/simple)

---

# 🔬 Part 1 — Research Experiment

This section begins the experiment used for the research paper.

The workflow evaluates a **two-qubit Variational Quantum Classifier (VQC)** for binary intent classification using the **Greeting** and **Farewell** classes from the **Tanaos synthetic intent classifier dataset**.

| Stage | Description |
|---|---|
| Data preparation | Load the dataset and keep only the target intent classes. |
| Feature processing | Convert text to TF-IDF vectors and reduce them to two PCA components. |
| Baseline model | Train and evaluate a clean two-qubit VQC. |
| Noise testing | Evaluate the model under depolarizing noise and finite shot counts. |
| Error mitigation | Apply Zero-Noise Extrapolation. |
| Monte Carlo analysis | Repeat trials and summarize mean accuracy and variance. |

## 1.1 Research Configuration

This section defines the settings used throughout the experiment.

| Setting | Purpose |
|---|---|
| Target labels | Restrict the task to `Greeting` and `Farewell`. |
| PCA components | Reduce text features to two values for the two-qubit circuit. |
| Random seed | Keep runs reproducible where possible. |
| Noise levels | Control the amount of depolarizing noise during simulation. |
| Shot counts | Test how finite measurement sampling affects results. |
| Monte Carlo trials | Repeat the experiment to estimate accuracy variance. |

The goal is not to build a large classifier. The goal is to observe how a small VQC behaves under controlled noise and sampling conditions.

In [ ]:
%%writefile src/config.py

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ExperimentConfig:
    """
    Central configuration for the VQC intent classification experiment.

    This keeps the main experiment settings in one place so the notebook and
    source files can reuse the same labels, seeds, feature dimensions, noise
    levels, shot counts, and trial counts.
    """

    # Project paths
    project_root: Path = Path.cwd()
    data_dir: Path = Path.cwd() / "data"
    results_dir: Path = Path.cwd() / "results"

    # Dataset settings
    dataset_name: str = "tanaos/synthetic-intent-classifier-dataset-v1"
    text_column: str = "text"
    label_column: str = "label"

    greeting_label: int = 0
    farewell_label: int = 1

    greeting_name: str = "Greeting"
    farewell_name: str = "Farewell"

    samples_per_label: int = 500
    test_size: float = 0.20
    balance_classes: bool = True

    # Reproducibility
    base_seed: int = 42

    # Feature extraction
    tfidf_max_features: int = 500
    pca_components: int = 2

    # VQC architecture
    num_qubits: int = 2
    feature_map: str = "ZZFeatureMap"
    feature_map_reps: int = 1
    ansatz: str = "RealAmplitudes"
    ansatz_reps: int = 2
    trainable_parameters: int = 6
    optimizer: str = "COBYLA"
    maxiter: int = 100

    # Noise and sampling settings
    noise_levels: tuple[float, ...] = (0.00, 0.01, 0.03, 0.05, 0.07, 0.10)
    shot_counts: tuple[int, ...] = (100, 1000, 10000)

    # Zero-Noise Extrapolation settings
    zne_scale_factors: tuple[int, ...] = (1, 3, 5)

    # Monte Carlo settings
    monte_carlo_trials: int = 1000


CONFIG = ExperimentConfig()

LABEL_MAP = {
    CONFIG.greeting_label: CONFIG.greeting_name,
    CONFIG.farewell_label: CONFIG.farewell_name,
}

TARGET_LABELS = list(LABEL_MAP.keys())

In [ ]:
import importlib
import random

import numpy as np
import pandas as pd

import src.config
importlib.reload(src.config)

from src.config import CONFIG, LABEL_MAP, TARGET_LABELS


# Create local output folders if they do not exist.
CONFIG.data_dir.mkdir(parents=True, exist_ok=True)
CONFIG.results_dir.mkdir(parents=True, exist_ok=True)

# Set base random seeds for reproducibility.
random.seed(CONFIG.base_seed)
np.random.seed(CONFIG.base_seed)

config_table = pd.DataFrame(
    {
        "Setting": [
            "Dataset",
            "Target labels",
            "Samples per label",
            "Test size",
            "Balance classes",
            "TF-IDF max features",
            "PCA components",
            "Number of qubits",
            "Feature map",
            "Feature map repetitions",
            "Ansatz",
            "Ansatz repetitions",
            "Trainable parameters",
            "Optimizer",
            "Max iterations",
            "Noise levels",
            "Shot counts",
            "ZNE scale factors",
            "Monte Carlo trials",
            "Base seed",
        ],
        "Value": [
            CONFIG.dataset_name,
            ", ".join(f"{key}: {value}" for key, value in LABEL_MAP.items()),
            CONFIG.samples_per_label,
            CONFIG.test_size,
            CONFIG.balance_classes,
            CONFIG.tfidf_max_features,
            CONFIG.pca_components,
            CONFIG.num_qubits,
            CONFIG.feature_map,
            CONFIG.feature_map_reps,
            CONFIG.ansatz,
            CONFIG.ansatz_reps,
            CONFIG.trainable_parameters,
            CONFIG.optimizer,
            CONFIG.maxiter,
            CONFIG.noise_levels,
            CONFIG.shot_counts,
            CONFIG.zne_scale_factors,
            CONFIG.monte_carlo_trials,
            CONFIG.base_seed,
        ],
    }
)

config_table

---

## 1.2 Dataset Loading

This section defines the dataset loading utility used by the experiment.

The loader prepares the binary intent classification task by filtering the Tanaos synthetic intent classifier dataset to the selected target labels:

| Label | Intent |
|---:|---|
| `0` | `Greeting` |
| `1` | `Farewell` |

The loader should handle three tasks:

| Task | Purpose |
|---|---|
| Filter labels | Keep only the intent classes used in this experiment. |
| Balance classes | Use the same number of samples per class when enabled. |
| Split dataset | Create reproducible training and testing sets. |

The output of this section will be used by the feature extraction stage.

In [ ]:
%%writefile src/data_loader.py
from __future__ import annotations

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

from src.config import CONFIG, LABEL_MAP, TARGET_LABELS


def fetch_intent_data(
    labels: list[int] | None = None,
    samples_per_label: int | None = None,
    test_size: float | None = None,
    random_state: int | None = None,
    balance_classes: bool | None = None,
):
    """
    Load the Tanaos synthetic intent classifier dataset and prepare the
    Greeting/Farewell subset for binary intent classification.

    Parameters default to values defined in src.config.CONFIG.
    """

    labels = TARGET_LABELS if labels is None else labels
    samples_per_label = CONFIG.samples_per_label if samples_per_label is None else samples_per_label
    test_size = CONFIG.test_size if test_size is None else test_size
    random_state = CONFIG.base_seed if random_state is None else random_state
    balance_classes = CONFIG.balance_classes if balance_classes is None else balance_classes

    print(f"Requesting dataset: tanaos/synthetic-intent-classifier-dataset-v1")

    raw_dataset = load_dataset(
        "tanaos/synthetic-intent-classifier-dataset-v1",
        split="train",
    )

    df = raw_dataset.to_pandas()

    # Standardize possible label column names.
    if "intent" in df.columns:
        df = df.rename(columns={"intent": "label"})
    elif "labels" in df.columns:
        df = df.rename(columns={"labels": "label"})

    if "label" not in df.columns:
        raise KeyError(
            f"Could not find a label column. Available columns: {list(df.columns)}"
        )

    if CONFIG.text_column not in df.columns:
        raise KeyError(
            f"Could not find text column '{CONFIG.text_column}'. "
            f"Available columns: {list(df.columns)}"
        )

    # Keep only the target labels from the research configuration.
    df_filtered = df[df["label"].isin(labels)].copy()

    if df_filtered.empty:
        raise ValueError(f"No rows found for target labels: {labels}")

    # Original class counts before balancing.
    original_counts = (
        df_filtered["label"]
        .value_counts()
        .sort_index()
        .rename(index=LABEL_MAP)
    )

    # Balance the selected classes if enabled.
    if balance_classes:
        balanced_frames = []

        for label, group in df_filtered.groupby("label"):
            n_samples = min(len(group), samples_per_label)

            sampled_group = group.sample(
                n=n_samples,
                random_state=random_state,
            )

            balanced_frames.append(sampled_group)

        final_df = pd.concat(balanced_frames).reset_index(drop=True)
    else:
        final_df = df_filtered.reset_index(drop=True)

    # Final class counts after optional balancing.
    final_counts = (
        final_df["label"]
        .value_counts()
        .sort_index()
        .rename(index=LABEL_MAP)
    )

    train_df, test_df = train_test_split(
        final_df,
        test_size=test_size,
        stratify=final_df["label"],
        random_state=random_state,
    )

    train_counts = (
        train_df["label"]
        .value_counts()
        .sort_index()
        .rename(index=LABEL_MAP)
    )

    test_counts = (
        test_df["label"]
        .value_counts()
        .sort_index()
        .rename(index=LABEL_MAP)
    )

    summary = {
        "original_counts": original_counts,
        "final_counts": final_counts,
        "train_counts": train_counts,
        "test_counts": test_counts,
        "total_samples": len(final_df),
        "train_samples": len(train_df),
        "test_samples": len(test_df),
        "test_size": test_size,
        "labels": labels,
        "balance_classes": balance_classes,
        "samples_per_label": samples_per_label,
        "random_state": random_state,
    }

    print(f"Success: {len(train_df)} training and {len(test_df)} testing samples retrieved.")

    return train_df, test_df, summary

In [ ]:
import importlib

import pandas as pd

import src.data_loader
importlib.reload(src.data_loader)

from src.config import CONFIG, LABEL_MAP, TARGET_LABELS
from src.data_loader import fetch_intent_data


train_df, test_df, dataset_summary = fetch_intent_data(
    labels=TARGET_LABELS,
    samples_per_label=CONFIG.samples_per_label,
    test_size=CONFIG.test_size,
    random_state=CONFIG.base_seed,
    balance_classes=CONFIG.balance_classes,
)

class_summary = pd.DataFrame(
    {
        "Original Dataset": dataset_summary["original_counts"],
        "After Balancing": dataset_summary["final_counts"],
        "Training Set": dataset_summary["train_counts"],
        "Testing Set": dataset_summary["test_counts"],
    }
).fillna(0).astype(int)

print("\n--- Dataset Summary for Paper ---")
print(f"Total Samples           : {dataset_summary['total_samples']}")
print(f"Training Samples        : {dataset_summary['train_samples']}")
print(f"Testing Samples         : {dataset_summary['test_samples']}")
print(f"Training/Testing Split  : {int((1 - CONFIG.test_size) * 100)}/{int(CONFIG.test_size * 100)}")
print(f"Target Labels           : {LABEL_MAP}")
print(f"Samples per Label       : {CONFIG.samples_per_label}")
print(f"Balance Classes         : {CONFIG.balance_classes}")
print(f"Status                  : Ready for Feature Extraction")

class_summary

### 1.2.1 Dataset Loading Notes

The experiment uses only two intent classes so the classification task remains compatible with the two-qubit VQC design.

Balancing the selected classes helps prevent the model from favoring one intent simply because it appears more often in the dataset. The train/test split is controlled by the base seed defined in the research configuration.

---

## 1.3 Text Preprocessing

This section prepares the text utterances before feature extraction.

The dataset contains short intent-classification examples. Before converting them into numerical features, the text is normalized so the feature extraction stage receives a cleaner input representation.

| Step | Purpose |
|---|---|
| Lowercasing | Reduces duplicate word forms caused by capitalization. |
| Token cleanup | Removes unnecessary symbols or unstable text artifacts. |
| Stopword handling | Removes common words when useful for the feature pipeline. |

The goal is not to perform deep linguistic analysis. The preprocessing step only prepares the text for the TF-IDF and PCA pipeline used in the next section.

---

## 1.4 Feature Extraction

This section converts text into numerical features that can be used by the two-qubit VQC.

The feature extraction pipeline uses two stages:

| Stage | Output |
|---|---|
| `TF-IDF` | Converts each utterance into a weighted text vector. |
| `PCA` | Reduces the TF-IDF vector space to two numerical components. |

The two PCA components are used as the input features for the two-qubit quantum circuit.

> The dimensionality reduction is intentional. Some information is lost, but the reduced feature space keeps the input compatible with the two-qubit architecture used in this study.

In [ ]:
%%writefile src/nlp_utils.py
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
import pandas as pd

from src.config import CONFIG

# Ensure NLTK resources are available locally
nltk.download('stopwords', quiet=True)

def process_features(train_df, test_df, n_components=None, max_features=None):
    """
    Transforms raw text into a reduced numerical space for Quantum mapping.
    Uses TF-IDF for vectorization and PCA for dimensionality reduction.

    Parameters default to values defined in src.config.CONFIG.
    """
    n_components = CONFIG.pca_components if n_components is None else n_components
    max_features = CONFIG.tfidf_max_features if max_features is None else max_features

    print(f"🛠️ Vectorizing text and reducing to {n_components} components...")
    
    # 1. Initialize TF-IDF with English stopword removal
    # We limit the feature count to maintain a manageable dense matrix
    stop_words = list(stopwords.words('english'))
    vectorizer = TfidfVectorizer(stop_words=stop_words, max_features=max_features)
    
    # 2. Fit and transform training data; transform test data using the training vocabulary
    # This prevents 'data leakage' from the test set into our feature space
    X_train_tfidf = vectorizer.fit_transform(train_df[CONFIG.text_column]).toarray()
    X_test_tfidf = vectorizer.transform(test_df[CONFIG.text_column]).toarray()
    
    # 3. Principal Component Analysis (PCA)
    # Reducing the semantic space to match our target Qubit count
    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train_tfidf)
    X_test_pca = pca.transform(X_test_tfidf)

    pca_variance = float(sum(pca.explained_variance_ratio_))
    
    print(f"✅ Feature Engineering Complete.")
    print(f"PCA Variance Explained: {pca_variance * 100:.2f}%")
    
    return X_train_pca, X_test_pca, vectorizer, pca, pca_variance

In [ ]:
import importlib

import src.nlp_utils
importlib.reload(src.nlp_utils)

from src.config import CONFIG
from src.nlp_utils import process_features

# Extract labels as standard numerical arrays
y_train = train_df[CONFIG.label_column].values
y_test = test_df[CONFIG.label_column].values

# Run the NLP pipeline to get 2-dimensional coordinates
X_train, X_test, vectorizer, pca, pca_variance = process_features(
    train_df,
    test_df,
    n_components=CONFIG.pca_components,
    max_features=CONFIG.tfidf_max_features,
)

# Verify the shape: (samples, 2)
print(f"\nTraining Data Shape: {X_train.shape}")
print(f"Testing Data Shape : {X_test.shape}")
print(f"PCA Variance       : {pca_variance * 100:.2f}%")

print("\nFirst 5 Training Vectors (Quantum Ready):")
print(X_train[:5])

### 1.4.1 Feature Extraction Notes

The output of this section should confirm that the text data has been converted into a two-dimensional numerical representation.

The important checks are:

| Check | Expected result |
|---|---|
| Training feature shape | Two columns, matching the two PCA components. |
| Testing feature shape | Two columns, matching the two PCA components. |
| PCA explained variance | Shows how much TF-IDF variance is retained after reduction. |

These features are passed to the VQC architecture in the next section.

In [ ]:
%%writefile src/quantum_engine.py
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit.primitives import StatevectorSampler

from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_algorithms.optimizers import COBYLA

from src.config import CONFIG


def parity(bitstring_as_int: int) -> int:
    """
    Map a measured computational-basis state to a binary class using parity.

    Even parity maps to class 0.
    Odd parity maps to class 1.

    This keeps the clean VQC and the noisy/ZNE evaluator aligned.
    """

    return bin(bitstring_as_int).count("1") % 2


def build_vqc():
    """
    Build the clean baseline Variational Quantum Classifier.

    This model uses StatevectorSampler, so it does not include depolarizing
    noise, finite-shot sampling, or error mitigation. The trained parameters
    from this clean model are reused later for fixed-model noisy evaluation.

    The VQC uses an explicit parity interpretation so that its class mapping
    matches the manual noisy/ZNE evaluator.
    """

    feature_map = ZZFeatureMap(
        feature_dimension=CONFIG.pca_components,
        reps=CONFIG.feature_map_reps,
    )

    ansatz = RealAmplitudes(
        num_qubits=CONFIG.num_qubits,
        reps=CONFIG.ansatz_reps,
    )

    optimizer = COBYLA(
        maxiter=CONFIG.maxiter,
    )

    sampler = StatevectorSampler()

    vqc = VQC(
        sampler=sampler,
        feature_map=feature_map,
        ansatz=ansatz,
        optimizer=optimizer,
        interpret=parity,
        output_shape=2,
    )

    return vqc, feature_map, ansatz


def train_vqc(X_train, y_train):
    """
    Train the clean baseline VQC using the processed training features.
    """

    vqc, feature_map, ansatz = build_vqc()

    print("Building clean baseline VQC...")
    print(f"Feature map  : {CONFIG.feature_map}")
    print(f"Feature reps : {CONFIG.feature_map_reps}")
    print(f"Ansatz       : {CONFIG.ansatz}")
    print(f"Ansatz reps  : {CONFIG.ansatz_reps}")
    print(f"Qubits       : {CONFIG.num_qubits}")
    print(f"Parameters   : {len(ansatz.parameters)}")
    print(f"Optimizer    : {CONFIG.optimizer}")
    print(f"Max iter     : {CONFIG.maxiter}")
    print("Interpret    : parity")
    print("Output shape : 2")

    print("\nTraining clean baseline VQC...")
    vqc.fit(X_train, y_train)

    print("Clean baseline VQC training complete.")

    return vqc, feature_map, ansatz

### 1.5 VQC Architecture

This section defines the shared two-qubit Variational Quantum Classifier architecture used for both the clean baseline and noisy simulations.

The model uses a `ZZFeatureMap` to encode the two PCA-reduced text features into a quantum feature space, followed by a `RealAmplitudes` ansatz as the trainable variational circuit. The classifier is optimized using COBYLA.

For the clean baseline, the model uses Qiskit's `StatevectorSampler`. For noisy evaluation, the trained parameters from the clean VQC are reused inside an Aer-based circuit simulation with a depolarizing noise model. This keeps the circuit architecture fixed while varying only the noise level, shot count, and Monte Carlo seed.

In [ ]:
import importlib

import matplotlib.pyplot as plt

import src.quantum_engine
importlib.reload(src.quantum_engine)

from src.config import CONFIG
from src.quantum_engine import train_vqc


vqc_model, feature_map, ansatz = train_vqc(X_train, y_train)

print("\nClean baseline VQC trained successfully.")
print(f"Feature map : {CONFIG.feature_map}")
print(f"Ansatz      : {CONFIG.ansatz}")
print(f"Qubits      : {CONFIG.num_qubits}")
print(f"Parameters  : {len(ansatz.parameters)}")

circuit_dir = CONFIG.results_dir / "circuits"
circuit_dir.mkdir(parents=True, exist_ok=True)

feature_map_circuit = feature_map.decompose()
ansatz_circuit = ansatz.decompose()
full_vqc_circuit = feature_map.compose(ansatz).decompose()

print("\nFeature Map Circuit:")
feature_map_fig = feature_map_circuit.draw(output="mpl", fold=-1)
feature_map_fig.savefig(
    circuit_dir / "feature_map.svg",
    format="svg",
    bbox_inches="tight",
)
display(feature_map_fig)
plt.close(feature_map_fig)

print("\nAnsatz Circuit:")
ansatz_fig = ansatz_circuit.draw(output="mpl", fold=-1)
ansatz_fig.savefig(
    circuit_dir / "ansatz.svg",
    format="svg",
    bbox_inches="tight",
)
display(ansatz_fig)
plt.close(ansatz_fig)

print("\nCombined VQC Circuit:")
full_vqc_fig = full_vqc_circuit.draw(output="mpl", fold=-1)
full_vqc_fig.savefig(
    circuit_dir / "full_vqc_circuit.svg",
    format="svg",
    bbox_inches="tight",
)
display(full_vqc_fig)
plt.close(full_vqc_fig)

---

## 1.6 Baseline Statevector Evaluation

This section evaluates the trained VQC without hardware noise or finite-shot sampling.

The baseline result is used as the reference point for later noisy simulations.

| Output | Purpose |
|---|---|
| Accuracy | Measures clean test-set performance. |
| Classification report | Shows class-level precision, recall, and F1-score. |
| Confusion matrix | Shows how Greeting and Farewell predictions are distributed. |

This section does not apply depolarizing noise, Zero-Noise Extrapolation, or Monte Carlo trials yet.

In [ ]:
%%writefile src/evaluation.py
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

from src.config import CONFIG, LABEL_MAP


def evaluate_classifier(model, X_test, y_test, title="Confusion Matrix"):
    """
    Evaluate a trained classifier using accuracy, a classification report,
    and a confusion matrix.
    """

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=[LABEL_MAP[label] for label in sorted(LABEL_MAP.keys())],
        )
    )

    labels = sorted(LABEL_MAP.keys())
    label_names = [LABEL_MAP[label] for label in labels]

    matrix = confusion_matrix(
        y_test,
        y_pred,
        labels=labels,
    )

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=label_names,
        yticklabels=label_names,
    )

    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()

    figure_dir = CONFIG.results_dir / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)

    safe_title = title.lower().replace(" ", "_")
    plt.savefig(
        figure_dir / f"{safe_title}.svg",
        format="svg",
        bbox_inches="tight",
    )

    plt.show()

    return {
        "accuracy": accuracy,
        "y_pred": y_pred,
        "confusion_matrix": matrix,
    }

In [ ]:
import importlib

import src.evaluation
importlib.reload(src.evaluation)

from src.evaluation import evaluate_classifier


baseline_results = evaluate_classifier(
    vqc_model,
    X_test,
    y_test,
    title="Clean Baseline VQC Confusion Matrix",
)

---

## 1.7 Noisy Simulation

This section introduces controlled depolarizing noise and finite measurement shots.

The clean baseline in the previous section used a statevector sampler, which does not include measurement noise or hardware-like circuit noise. This section prepares the noise simulation layer that will later be used for Monte Carlo trials and Zero-Noise Extrapolation.

| Component | Purpose |
|---|---|
| Depolarizing noise | Simulates random quantum-state disturbance after gates. |
| Shot count | Controls how many measurement samples are taken from the circuit. |
| Random seed | Keeps noisy simulation runs reproducible where possible. |

The first step is a smoke test: create a small circuit, apply depolarizing noise, and inspect the output counts.

In [ ]:
%%writefile src/noise_simulation.py
from __future__ import annotations

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

from src.config import CONFIG


def build_depolarizing_noise_model(noise_level: float) -> NoiseModel | None:
    """
    Build a depolarizing noise model for simulation.

    A noise level of 0.0 returns None, which represents the noiseless simulator.
    For nonzero noise levels, single-qubit and two-qubit depolarizing errors
    are attached to common circuit gates.
    """

    if noise_level <= 0.0:
        return None

    if not 0.0 <= noise_level <= 1.0:
        raise ValueError("noise_level must be between 0.0 and 1.0")

    noise_model = NoiseModel()

    single_qubit_error = depolarizing_error(noise_level, 1)
    two_qubit_error = depolarizing_error(noise_level, 2)

    # Common one-qubit gates after circuit decomposition/transpilation.
    one_qubit_gates = [
        "id",
        "x",
        "sx",
        "h",
        "rx",
        "ry",
        "rz",
    ]

    # Common two-qubit gates used after decomposition/transpilation.
    two_qubit_gates = [
        "cx",
        "cz",
    ]

    noise_model.add_all_qubit_quantum_error(single_qubit_error, one_qubit_gates)
    noise_model.add_all_qubit_quantum_error(two_qubit_error, two_qubit_gates)

    return noise_model


def run_noisy_counts(
    circuit: QuantumCircuit,
    noise_level: float,
    shots: int,
    seed: int | None = None,
) -> dict:
    """
    Run a quantum circuit using an Aer simulator with optional depolarizing noise.

    The circuit is measured before execution. The function returns raw measurement
    counts such as {'00': 512, '11': 488}.
    """

    measured_circuit = circuit.copy()
    measured_circuit.measure_all()

    noise_model = build_depolarizing_noise_model(noise_level)

    simulator = AerSimulator(
        noise_model=noise_model,
        seed_simulator=CONFIG.base_seed if seed is None else seed,
    )

    transpiled_circuit = transpile(
        measured_circuit,
        simulator,
        seed_transpiler=CONFIG.base_seed if seed is None else seed,
    )

    result = simulator.run(
        transpiled_circuit,
        shots=shots,
    ).result()

    return result.get_counts()

In [ ]:
import importlib

from qiskit import QuantumCircuit

import src.noise_simulation
importlib.reload(src.noise_simulation)

from src.config import CONFIG
from src.noise_simulation import run_noisy_counts


# Small two-qubit circuit used only to verify the noisy simulator.
test_circuit = QuantumCircuit(CONFIG.num_qubits)
test_circuit.h(0)
test_circuit.cx(0, 1)

smoke_test_dir = CONFIG.results_dir / "circuits"
smoke_test_dir.mkdir(parents=True, exist_ok=True)

print("Test Circuit:")
test_circuit_fig = test_circuit.draw(output="mpl", fold=-1)
test_circuit_fig.savefig(
    smoke_test_dir / "noise_smoke_test_circuit.svg",
    format="svg",
    bbox_inches="tight",
)
display(test_circuit_fig)
plt.close(test_circuit_fig)

noise_smoke_results = {}

for noise_level in CONFIG.noise_levels:
    counts = run_noisy_counts(
        circuit=test_circuit,
        noise_level=noise_level,
        shots=CONFIG.shot_counts[1],
        seed=CONFIG.base_seed,
    )

    noise_smoke_results[noise_level] = counts

noise_smoke_results

### 1.7.1 Noisy Simulation Smoke Test Notes

The smoke test confirms that the depolarizing noise model is active.

In the noiseless case, the test circuit mostly produces `00` and `11`, which is expected for the Bell-state circuit. As the depolarizing noise level increases, the incorrect states `01` and `10` appear more often.

This confirms that the simulator can inject controlled noise before it is connected to the VQC evaluation pipeline.

### 1.7.2 Fixed-Model Noisy VQC Evaluation

This section evaluates the already-trained clean VQC under simulated quantum noise.

The VQC is not retrained for each noise and shot configuration. Instead, the trained parameters from the clean baseline are reused while the simulator noise level and measurement shot count are varied.

| Item | Meaning |
|---|---|
| Fixed model | The clean VQC trained in Section 1.5 is reused. |
| Noise level | Depolarizing noise applied during circuit execution. |
| Shot count | Number of measurement samples used during evaluation. |
| Output | Accuracy of the fixed trained model under each noisy condition. |

This setup better isolates the effect of depolarizing noise and finite-shot sampling.

---

## 1.8 Zero-Noise Extrapolation

This section applies Zero-Noise Extrapolation (ZNE) to the fixed trained VQC.

The model is still not retrained. For each base noise level, shot count, and Monte Carlo seed, the same trained VQC is evaluated at scaled noise levels defined by `CONFIG.zne_scale_factors`. A linear extrapolation is then used to estimate the accuracy at the zero-noise limit.

In [ ]:
%%writefile src/zne_evaluation.py
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Any

import numpy as np
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from sklearn.metrics import accuracy_score

from src.noise_simulation import build_depolarizing_noise_model


@dataclass(frozen=True)
class ZNERunConfig:
    """
    Configuration for one fixed-model ZNE evaluation run.
    """

    base_noise_level: float
    shots: int
    seed: int
    scale_factors: tuple[int, ...]


def get_trained_weights(vqc_model) -> np.ndarray:
    """
    Extract trained weights from a fitted Qiskit Machine Learning VQC.
    """

    if hasattr(vqc_model, "weights") and vqc_model.weights is not None:
        return np.asarray(vqc_model.weights, dtype=float)

    if hasattr(vqc_model, "fit_result") and vqc_model.fit_result is not None:
        return np.asarray(vqc_model.fit_result.x, dtype=float)

    if hasattr(vqc_model, "_fit_result") and vqc_model._fit_result is not None:
        return np.asarray(vqc_model._fit_result.x, dtype=float)

    raise AttributeError(
        "Could not extract trained weights from the VQC model. "
        "Make sure the VQC has been fitted before noisy or ZNE evaluation."
    )


def counts_to_binary_label(counts: dict[str, int]) -> int:
    """
    Convert measured bitstring counts into a binary class label.

    The same parity-style interpretation used by Qiskit's default binary VQC
    interpretation is applied here: even parity maps to class 0, odd parity
    maps to class 1.
    """

    class_counts = {
        0: 0,
        1: 0,
    }

    for bitstring, count in counts.items():
        cleaned = bitstring.replace(" ", "")
        parity = sum(int(bit) for bit in cleaned) % 2
        class_counts[parity] += count

    if class_counts[0] >= class_counts[1]:
        return 0

    return 1


def build_bound_circuits(
    parameterized_circuit,
    feature_map,
    ansatz,
    trained_weights: np.ndarray,
    X,
):
    """
    Bind test features and trained VQC weights to the parameterized circuit.

    The VQC is not retrained. Each test sample receives its own feature values,
    while all samples reuse the same trained ansatz weights.
    """

    input_parameters = list(feature_map.parameters)
    weight_parameters = list(ansatz.parameters)

    if len(input_parameters) != X.shape[1]:
        raise ValueError(
            f"Expected {len(input_parameters)} input features, "
            f"but received {X.shape[1]}."
        )

    if len(weight_parameters) != len(trained_weights):
        raise ValueError(
            f"Expected {len(weight_parameters)} trained weights, "
            f"but received {len(trained_weights)}."
        )

    bound_circuits = []

    for sample in X:
        parameter_values = {}

        for parameter, value in zip(input_parameters, sample):
            parameter_values[parameter] = float(value)

        for parameter, value in zip(weight_parameters, trained_weights):
            parameter_values[parameter] = float(value)

        bound_circuit = parameterized_circuit.assign_parameters(
            parameter_values,
            inplace=False,
        )

        bound_circuits.append(bound_circuit)

    return bound_circuits


def evaluate_fixed_vqc_under_noise(
    vqc_model,
    feature_map,
    ansatz,
    X_test,
    y_test,
    noise_level: float,
    shots: int,
    seed: int,
) -> float:
    """
    Evaluate a trained VQC under depolarizing noise and finite shots.

    The VQC is not retrained. The trained parameters are reused, and only the
    simulator noise level, shot count, and seed are changed.
    """

    trained_weights = get_trained_weights(vqc_model)

    full_circuit = feature_map.compose(ansatz)
    full_circuit.measure_all()
    full_circuit = full_circuit.decompose()

    noise_model = build_depolarizing_noise_model(noise_level)

    backend = AerSimulator(
        noise_model=noise_model,
        seed_simulator=seed,
    )

    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        seed_transpiler=seed,
    )

    transpiled_circuit = pass_manager.run(full_circuit)

    bound_circuits = build_bound_circuits(
        parameterized_circuit=transpiled_circuit,
        feature_map=feature_map,
        ansatz=ansatz,
        trained_weights=trained_weights,
        X=X_test,
    )

    result = backend.run(
        bound_circuits,
        shots=shots,
    ).result()

    predictions = []

    for index in range(len(bound_circuits)):
        counts = result.get_counts(index)
        prediction = counts_to_binary_label(counts)
        predictions.append(prediction)

    predictions = np.asarray(predictions)
    accuracy = accuracy_score(y_test, predictions)

    return float(accuracy)


def extrapolate_zero_noise(
    scaled_noise_levels: list[float],
    scaled_accuracies: list[float],
) -> float:
    """
    Estimate the zero-noise accuracy using linear extrapolation.

    The x-axis is the scaled noise level.
    The y-axis is the observed accuracy at that scaled noise level.
    The extrapolated value is the fitted intercept at noise = 0.
    """

    x = np.asarray(scaled_noise_levels, dtype=float)
    y = np.asarray(scaled_accuracies, dtype=float)

    if len(set(x)) == 1:
        return float(y[0])

    slope, intercept = np.polyfit(x, y, deg=1)

    return float(np.clip(intercept, 0.0, 1.0))


def evaluate_zne_fixed_vqc(
    vqc_model,
    feature_map,
    ansatz,
    X_test,
    y_test,
    run_config: ZNERunConfig,
) -> dict[str, Any]:
    """
    Evaluate a fixed trained VQC using Zero-Noise Extrapolation.

    The trained VQC is not retrained. The same trained parameters are evaluated
    at scaled depolarizing noise levels, then linearly extrapolated to estimate
    the zero-noise accuracy.
    """

    scaled_noise_levels = []
    scaled_accuracies = []

    for scale_factor in run_config.scale_factors:
        scaled_noise_level = run_config.base_noise_level * scale_factor
        scaled_noise_level = min(scaled_noise_level, 1.0)

        accuracy = evaluate_fixed_vqc_under_noise(
            vqc_model=vqc_model,
            feature_map=feature_map,
            ansatz=ansatz,
            X_test=X_test,
            y_test=y_test,
            noise_level=scaled_noise_level,
            shots=run_config.shots,
            seed=run_config.seed,
        )

        scaled_noise_levels.append(scaled_noise_level)
        scaled_accuracies.append(accuracy)

    raw_accuracy = scaled_accuracies[0]

    zne_accuracy = extrapolate_zero_noise(
        scaled_noise_levels=scaled_noise_levels,
        scaled_accuracies=scaled_accuracies,
    )

    output = asdict(run_config)
    output["raw_accuracy"] = float(raw_accuracy)
    output["zne_accuracy"] = float(zne_accuracy)
    output["zne_delta"] = float(zne_accuracy - raw_accuracy)
    output["scaled_noise_levels"] = tuple(scaled_noise_levels)
    output["scaled_accuracies"] = tuple(scaled_accuracies)

    return output


def run_zne_sweep(
    jobs: list[ZNERunConfig],
    vqc_model,
    feature_map,
    ansatz,
    X_test,
    y_test,
) -> list[dict[str, Any]]:
    """
    Run fixed-model ZNE evaluation across all configured jobs.
    """

    rows = []

    print(f"Running {len(jobs)} ZNE evaluation(s)...")

    for index, job in enumerate(jobs, start=1):
        print(
            f"\n[{index}/{len(jobs)}] "
            f"base_noise={job.base_noise_level:.2f}, "
            f"shots={job.shots}, "
            f"seed={job.seed}, "
            f"scales={job.scale_factors}"
        )

        result = evaluate_zne_fixed_vqc(
            vqc_model=vqc_model,
            feature_map=feature_map,
            ansatz=ansatz,
            X_test=X_test,
            y_test=y_test,
            run_config=job,
        )

        rows.append(result)

        print(
            f"done | base_noise={result['base_noise_level']:.2f}, "
            f"shots={result['shots']}, "
            f"seed={result['seed']}, "
            f"raw={result['raw_accuracy']:.4f}, "
            f"zne={result['zne_accuracy']:.4f}, "
            f"delta={result['zne_delta']:.4f}"
        )

    return rows

In [ ]:
import importlib

import pandas as pd

import src.zne_evaluation
importlib.reload(src.zne_evaluation)

from src.config import CONFIG
from src.zne_evaluation import ZNERunConfig, run_zne_sweep


zne_jobs = [
    ZNERunConfig(
        base_noise_level=noise_level,
        shots=shots,
        seed=CONFIG.base_seed + trial,
        scale_factors=CONFIG.zne_scale_factors,
    )
    for noise_level in CONFIG.noise_levels
    for shots in CONFIG.shot_counts
    for trial in range(CONFIG.monte_carlo_trials)
]

zne_rows = run_zne_sweep(
    jobs=zne_jobs,
    vqc_model=vqc_model,
    feature_map=feature_map,
    ansatz=ansatz,
    X_test=X_test,
    y_test=y_test,
)

zne_results = (
    pd.DataFrame(zne_rows)
    .sort_values(["base_noise_level", "shots", "seed"])
    .reset_index(drop=True)
)

zne_results

In [ ]:
zne_summary = (
    zne_results
    .groupby(["base_noise_level", "shots"], as_index=False)
    .agg(
        mean_raw_accuracy=("raw_accuracy", "mean"),
        mean_zne_accuracy=("zne_accuracy", "mean"),
        variance_raw_accuracy=("raw_accuracy", "var"),
        variance_zne_accuracy=("zne_accuracy", "var"),
        std_raw_accuracy=("raw_accuracy", "std"),
        std_zne_accuracy=("zne_accuracy", "std"),
        mean_zne_delta=("zne_delta", "mean"),
        min_zne_accuracy=("zne_accuracy", "min"),
        max_zne_accuracy=("zne_accuracy", "max"),
        trials=("zne_accuracy", "count"),
    )
)

zne_summary

In [ ]:
import matplotlib.pyplot as plt
from datetime import datetime


plot_dir = CONFIG.results_dir / "figures"
plot_dir.mkdir(parents=True, exist_ok=True)

plot_df = zne_summary.copy()

# Unique filename so figures from different runs do not overwrite each other.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

# Compute tighter y-axis bounds from the actual plotted values.
raw_min = plot_df["mean_raw_accuracy"].min()
raw_max = plot_df["mean_raw_accuracy"].max()
zne_min = plot_df["mean_zne_accuracy"].min()
zne_max = plot_df["mean_zne_accuracy"].max()

y_min = min(raw_min, zne_min)
y_max = max(raw_max, zne_max)

padding = 0.01
lower_bound = max(0.0, y_min - padding)
upper_bound = min(1.0, y_max + padding)

plt.figure(figsize=(11, 6.5))

for shots in sorted(plot_df["shots"].unique()):
    subset = plot_df[plot_df["shots"] == shots].sort_values("base_noise_level")

    plt.plot(
        subset["base_noise_level"],
        subset["mean_raw_accuracy"],
        marker="o",
        linestyle="--",
        linewidth=2,
        label=f"Raw accuracy — {shots} shots",
    )

    plt.plot(
        subset["base_noise_level"],
        subset["mean_zne_accuracy"],
        marker="o",
        linestyle="-",
        linewidth=2,
        label=f"ZNE accuracy — {shots} shots",
    )

plt.title("Raw vs ZNE Mean Accuracy Across Noise Levels")
plt.xlabel("Depolarizing Noise Level")
plt.ylabel("Mean Accuracy")

# Make the plot less visually compressed.
plt.ylim(lower_bound, upper_bound)

# Show only the actual tested noise levels.
plt.xticks(sorted(plot_df["base_noise_level"].unique()))

plt.grid(True, alpha=0.3)

# Move legend outside so it does not crowd the graph.
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))

plt.tight_layout()

mean_accuracy_plot_path = plot_dir / f"raw_vs_zne_mean_accuracy_{timestamp}.svg"

plt.savefig(
    mean_accuracy_plot_path,
    format="svg",
    bbox_inches="tight",
)

print(f"Saved figure: {mean_accuracy_plot_path}")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from datetime import datetime


plot_dir = CONFIG.results_dir / "figures"
plot_dir.mkdir(parents=True, exist_ok=True)

plot_df = zne_summary.copy()

# Unique filename so figures from different runs do not overwrite each other.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

# Compute tighter y-axis bounds from the actual plotted values.
raw_min = plot_df["variance_raw_accuracy"].min()
raw_max = plot_df["variance_raw_accuracy"].max()
zne_min = plot_df["variance_zne_accuracy"].min()
zne_max = plot_df["variance_zne_accuracy"].max()

y_min = min(raw_min, zne_min)
y_max = max(raw_max, zne_max)

padding = max((y_max - y_min) * 0.10, 0.00002)
lower_bound = max(0.0, y_min - padding)
upper_bound = y_max + padding

plt.figure(figsize=(11, 6.5))

for shots in sorted(plot_df["shots"].unique()):
    subset = plot_df[plot_df["shots"] == shots].sort_values("base_noise_level")

    plt.plot(
        subset["base_noise_level"],
        subset["variance_raw_accuracy"],
        marker="o",
        linestyle="--",
        linewidth=2,
        label=f"Raw variance — {shots} shots",
    )

    plt.plot(
        subset["base_noise_level"],
        subset["variance_zne_accuracy"],
        marker="o",
        linestyle="-",
        linewidth=2,
        label=f"ZNE variance — {shots} shots",
    )

plt.title("Raw vs ZNE Accuracy Variance Across Noise Levels")
plt.xlabel("Depolarizing Noise Level")
plt.ylabel("Accuracy Variance")

plt.ylim(lower_bound, upper_bound)

# Show only the actual tested noise levels.
plt.xticks(sorted(plot_df["base_noise_level"].unique()))
plt.gca().xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))

plt.grid(True, alpha=0.3)

# Move legend outside so it does not cover the plotted lines.
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))

plt.tight_layout()

variance_plot_path = plot_dir / f"raw_vs_zne_accuracy_variance_{timestamp}.svg"

plt.savefig(
    variance_plot_path,
    format="svg",
    bbox_inches="tight",
)

print(f"Saved figure: {variance_plot_path}")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from datetime import datetime


plot_dir = CONFIG.results_dir / "figures"
plot_dir.mkdir(parents=True, exist_ok=True)

plot_df = zne_summary.copy()

# Unique filename so figures from different runs do not overwrite each other.
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

# Compute tighter y-axis bounds from actual ZNE gains.
y_min = float(plot_df["mean_zne_delta"].min())
y_max = float(plot_df["mean_zne_delta"].max())

padding = max((y_max - y_min) * 0.10, 0.002)
lower_bound = y_min - padding
upper_bound = y_max + padding

plt.figure(figsize=(11, 6.5))

for shots in sorted(plot_df["shots"].unique()):
    subset = plot_df[plot_df["shots"] == shots].sort_values("base_noise_level")

    plt.plot(
        subset["base_noise_level"],
        subset["mean_zne_delta"],
        marker="o",
        linestyle="-",
        linewidth=2,
        label=f"ZNE gain — {shots} shots",
    )

plt.axhline(0.0, linestyle="--", linewidth=1)

plt.title("ZNE Accuracy Gain Across Noise Levels")
plt.xlabel("Depolarizing Noise Level")
plt.ylabel("Mean ZNE Accuracy Gain")

plt.ylim(lower_bound, upper_bound)

# Show only the actual tested noise levels.
plt.xticks(sorted(plot_df["base_noise_level"].unique()))
plt.gca().xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))

plt.grid(True, alpha=0.3)
plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))

plt.tight_layout()

zne_gain_plot_path = plot_dir / f"zne_accuracy_gain_{timestamp}.svg"

plt.savefig(
    zne_gain_plot_path,
    format="svg",
    bbox_inches="tight",
)

print(f"Saved figure: {zne_gain_plot_path}")

plt.show()

In [ ]:
# Save ZNE Monte Carlo raw results and summary table as unique CSV files.
# This prevents accidental overwriting if the notebook is run again.

from datetime import datetime

csv_dir = CONFIG.results_dir / "csv"
csv_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")

zne_results_path = csv_dir / f"zne_results_{timestamp}.csv"
zne_summary_path = csv_dir / f"zne_summary_{timestamp}.csv"

zne_results.to_csv(zne_results_path, index=False)
zne_summary.to_csv(zne_summary_path, index=False)

print("Saved Monte Carlo result files:")
print(f"- {zne_results_path}")
print(f"- {zne_summary_path}")

---

## 🔗 Project Repository

> 📦 **Link** — Source code, notebooks, and documentation.